# Notebook 3 — Fine-Tune YOLOv8 and Export to ONNX

**What you will do:**
1. Build a small 3-class dataset: `intact`, `damaged`, `tampered`.
2. Fine-tune YOLOv8n-cls on that dataset (CPU, ~5–15 min).
3. Export the best checkpoint to ONNX format.
4. Upload the ONNX model to your S3 bucket so it can be served by KServe.

> ⏱ **Timing:** Training 10 epochs on CPU typically takes 5–15 minutes.
> You can reduce `EPOCHS` to 5 for a quick test run.

In [ ]:
# ── Cell 0: Sync lab materials from GitHub ────────────────────────────────────
import subprocess, os, pathlib

REPO_URL   = 'https://github.com/faheemshai/1512_model_training.git'
LOCAL_PATH = os.path.expanduser('~/lab-materials')

if pathlib.Path(LOCAL_PATH, '.git').is_dir():
    r = subprocess.run(['git', '-C', LOCAL_PATH, 'pull', '--ff-only'],
                       capture_output=True, text=True)
    print('Repo up to date:', r.stdout.strip() or 'Already up to date.')
else:
    print('Cloning lab repo (first time, ~10 s)...')
    r = subprocess.run(['git', 'clone', REPO_URL, LOCAL_PATH],
                       capture_output=True, text=True)
    print(r.stderr.strip())

LAB = LOCAL_PATH
print(f'✅ Lab materials ready at: {LAB}')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
import os

EPOCHS      = 10        # increase to 30 for better accuracy
IMG_SIZE    = 64        # small image size keeps CPU training fast
BATCH_SIZE  = 16
MODEL_NAME  = 'image-classifier'

# S3 config — read from environment (injected by workbench data connection)
S3_ENDPOINT   = os.environ.get('AWS_S3_ENDPOINT',          'http://s3.openshift-storage.svc:80')
S3_ACCESS_KEY = os.environ.get('AWS_ACCESS_KEY_ID',        '')
S3_SECRET_KEY = os.environ.get('AWS_SECRET_ACCESS_KEY',    '')
S3_BUCKET     = os.environ.get('AWS_S3_BUCKET',            '')
NAMESPACE     = open('/var/run/secrets/kubernetes.io/serviceaccount/namespace').read().strip()

S3_MODEL_KEY  = f'models/{MODEL_NAME}/1/model.onnx'

print(f'Namespace  : {NAMESPACE}')
print(f'S3 bucket  : {S3_BUCKET}')
print(f'Model key  : {S3_MODEL_KEY}')
print(f'Epochs     : {EPOCHS}')

In [ ]:
# ── Build a 3-class dataset from real + augmented images ──────────────────────
# Uses the 3 real images in sample-images/ (from git repo) as seeds.
# Augmentation expands each to 10 images per class.
# No internet access required.

import pathlib, random
import numpy as np
from PIL import Image, ImageEnhance

DATASET_DIR = pathlib.Path('parcel-dataset')
CLASSES     = ['intact', 'damaged', 'tampered']

# Seed images — from the git repo
IMG_DIR     = pathlib.Path(LAB) / 'sample-images'
CLASS_SEEDS = {
    'intact'  : IMG_DIR / 'cardboard_box.jpg',
    'damaged' : IMG_DIR / 'damaged_package.jpg',
    'tampered': IMG_DIR / 'street_scene.jpg',
}

IMAGES_PER_CLASS = 10   # 8 train + 2 val

random.seed(42)
np.random.seed(42)

def augment(img, idx):
    img = img.convert('RGB')
    if idx % 2 == 0:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    if idx % 3 == 0:
        img = img.transpose(Image.FLIP_TOP_BOTTOM)
    img = ImageEnhance.Brightness(img).enhance(0.8 + (idx % 5) * 0.1)
    img = ImageEnhance.Contrast(img).enhance(0.85 + (idx % 4) * 0.1)
    img = ImageEnhance.Color(img).enhance(0.7 + (idx % 3) * 0.2)
    w, h = img.size
    margin = int(min(w, h) * 0.05 * (idx % 3))
    if margin > 0:
        img = img.crop((margin, margin, w - margin, h - margin))
    arr = np.array(img).astype(np.int16)
    arr += np.random.randint(-8, 8, arr.shape, dtype=np.int16)
    arr = np.clip(arr, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

for split in ['train', 'val']:
    for cls in CLASSES:
        (DATASET_DIR / split / cls).mkdir(parents=True, exist_ok=True)

for cls, seed_path in CLASS_SEEDS.items():
    seed_img = Image.open(seed_path).convert('RGB')
    imgs = [augment(seed_img, i) for i in range(IMAGES_PER_CLASS)]
    for i, img in enumerate(imgs[:8]):
        img.resize((IMG_SIZE, IMG_SIZE)).save(DATASET_DIR / 'train' / cls / f'{i:03d}.jpg')
    for i, img in enumerate(imgs[8:]):
        img.resize((IMG_SIZE, IMG_SIZE)).save(DATASET_DIR / 'val'   / cls / f'{i:03d}.jpg')

for split in ['train', 'val']:
    for cls in CLASSES:
        count = len(list((DATASET_DIR / split / cls).glob('*.jpg')))
        print(f'  {split}/{cls}: {count} images')

print('\n✅ Dataset ready — built from real images + augmentation (no internet needed)')

In [ ]:
# ── Fine-tune YOLOv8n-cls ─────────────────────────────────────────────────────
from ultralytics import YOLO
import time, pathlib

print(f'Starting fine-tuning: {EPOCHS} epochs, img_size={IMG_SIZE}, batch={BATCH_SIZE}')
print('⏱  This will take 5–15 minutes on CPU ...\n')

# Load weights from git repo — no internet download
WEIGHTS = str(pathlib.Path(LAB) / 'models' / 'yolov8n-cls.pt')
model   = YOLO(WEIGHTS)

t0 = time.time()
results = model.train(
    data    = str(DATASET_DIR),
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = BATCH_SIZE,
    device  = 'cpu',
    project = 'runs/classify',
    name    = 'parcel-ft',
    exist_ok= True,
    verbose = False,
)

elapsed = time.time() - t0
print(f'\n✅ Training complete in {elapsed/60:.1f} minutes')

In [ ]:
# ── Find best.pt (YOLO nests project dir when using relative path) ────────────
import glob as _glob
_candidates = _glob.glob('**/parcel-ft/weights/best.pt', recursive=True)
assert _candidates, 'best.pt not found — did training complete?'
BEST_PT = _candidates[0]
print(f'Best checkpoint found: {BEST_PT}')

# ── Validate: run best model against val images ───────────────────────────────
best_model = YOLO(BEST_PT)
val_results = best_model.val(data=str(DATASET_DIR), verbose=False)
print(f'Validation top-1 accuracy: {val_results.top1*100:.1f}%')
print(f'Validation top-5 accuracy: {val_results.top5*100:.1f}%')

In [ ]:
# ── Export to ONNX ────────────────────────────────────────────────────────────
print('Exporting to ONNX ...')
onnx_path = best_model.export(format='onnx', imgsz=IMG_SIZE, opset=13)
print(f'✅ ONNX model saved: {onnx_path}')

In [ ]:
# ── Upload ONNX model to S3 ───────────────────────────────────────────────────
import boto3
from botocore.client import Config

s3 = boto3.client(
    's3',
    endpoint_url          = S3_ENDPOINT,
    aws_access_key_id     = S3_ACCESS_KEY,
    aws_secret_access_key = S3_SECRET_KEY,
    region_name           = 'us-east-1',
    config                = Config(signature_version='s3v4'),
    verify                = False,
)

print(f'Uploading to s3://{S3_BUCKET}/{S3_MODEL_KEY} ...')
s3.upload_file(str(onnx_path), S3_BUCKET, S3_MODEL_KEY)

obj = s3.head_object(Bucket=S3_BUCKET, Key=S3_MODEL_KEY)
size_kb = obj['ContentLength'] / 1024
print(f'✅ Upload complete — {size_kb:.1f} KB at {S3_MODEL_KEY}')
print(f'\nModel path for KServe: models/{MODEL_NAME}/1/')
print('(KServe/OpenVINO expects model.onnx at path <prefix>/1/model.onnx)')

In [ ]:
# ── Save class names for use in Notebook 4 ───────────────────────────────────
import json
class_info = {'classes': CLASSES, 'model_key': S3_MODEL_KEY, 'bucket': S3_BUCKET}
with open('model-info.json', 'w') as f:
    json.dump(class_info, f, indent=2)
print('✅ Saved model-info.json')

print('\n══════════════════════════════════════════')
print(' Notebook 3 complete!')
print('══════════════════════════════════════════')
print(f' Classes   : {CLASSES}')
print(f' ONNX path : {onnx_path}')
print(f' S3 key    : {S3_MODEL_KEY}')
print('\n➡  Next: ask Bob to deploy the model with KServe (Step 6 in the lab guide)')